# E12: chạy lại OTI trên ImageNet bằng câu GPT mới, chia 3 phần

Mỗi phần nghịch đảo khoảng 33.000 ảnh ImageNet (một shard cũ `oti_B32_s0/s1/s2`) bằng bộ câu `P3_qwen15i_allf`.

| | A100 | L4 |
|---|---|---|
| Thời gian một phần | khoảng 12 giờ | khoảng 24 giờ |
| Đơn vị tính toán | khoảng 80 | khoảng 47 |

**Cách chạy một phần**

1. Thời gian chạy → Thay đổi loại thời gian chạy → GPU A100 (hoặc L4).
2. Ô 1: đặt `PART = 1`, `2` hoặc `3`.
3. Chạy ô 1 → 4 theo thứ tự. Ô 2 hỏi quyền Google Drive.
4. Để ô 4 chạy (giữ kết nối, in tiến độ mỗi 5 phút) và giữ tab mở.

**Ghi chú**

- Bị ngắt giữa chừng: chạy lại đúng phần đó, nó làm tiếp từ chỗ dừng (token được lưu mỗi 5 batch).
- Khi xong phần cuối cùng, phần đó tự gộp 3 phần, train lại φ (`phi_B32_P3oti`, khoảng 3 giờ trên A100), đánh giá có khoảng tin cậy và sinh file test `phiP3oti_cirr`, `phiP3oti_circo`.
- Log: `MyDrive/SEARLE/queue/q14_part<N>.log`.
- Có thể chạy các phần ở các phiên khác nhau, kể cả cùng lúc nếu có 2 runtime.


In [ ]:
# Ô 1: chọn phần và kiểm tra GPU
PART = 1   # 1, 2 hoặc 3
import subprocess
print('GPU:', subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                             capture_output=True, text=True).stdout.strip() or 'không có GPU')


In [ ]:
# Ô 2: gắn Drive và khởi động môi trường (code từ GitHub, venv, dữ liệu; khoảng 25 phút lần đầu)
from google.colab import drive; drive.mount('/content/drive')
import os, subprocess
if not os.path.exists('/content/boot.log'):
    subprocess.Popen('setsid nohup bash /content/drive/MyDrive/SEARLE/queue/boot.sh > /content/boot.log 2>&1 < /dev/null &',
                     shell=True)
    print('boot started')
else:
    print('boot already started')


In [ ]:
# Ô 3: chờ khởi động xong rồi chạy phần PART của q14 (chạy nền, dừng ô này không dừng q14)
import subprocess, time
while 'boot done' not in open('/content/boot.log', errors='replace').read():
    time.sleep(30)
cmd = (f"source /content/env.sh; git -C /content/XLA_SEARLE fetch -q origin && git -C /content/XLA_SEARLE reset -q --hard origin/main; "
       f"cp $SRC/colab/queue/q14.sh $DRIVE/queue/q14.sh; "
       f"pgrep -f 'q14.sh {PART}$' > /dev/null && echo 'q14 part {PART} already running' || "
       f"(setsid nohup bash $DRIVE/queue/q14.sh {PART} >> $DRIVE/queue/q14_part{PART}.log 2>&1 < /dev/null & echo 'q14 part {PART} started')")
print(subprocess.run(['bash', '-c', cmd], capture_output=True, text=True).stdout)


In [ ]:
# Ô 4: giữ kết nối + theo dõi (in mỗi 5 phút; dừng ô này không dừng q14)
import subprocess, time
from IPython.display import clear_output
S = f's{PART - 1}'
cmd = (f"D=/content/drive/MyDrive/SEARLE; date; tail -n 4 $D/queue/q14_part{PART}.log; "
       f"tail -c 2000 $D/results/E12.0-oti-{S}/stdout.log 2>/dev/null | tr '\\r' '\\n' | grep -a '%' | tail -n 1 | cut -c1-150; "
       f"tail -c 2000 $D/results/E12.2-train-phi_B32_P3oti/stdout.log 2>/dev/null | tr '\\r' '\\n' | grep -a '/100' | tail -n 1 | cut -c1-150; "
       "nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader")
for i in range(36 * 12):
    out = subprocess.run(['bash', '-c', cmd], capture_output=True, text=True, encoding='utf-8', errors='replace').stdout
    clear_output(wait=True); print(out, flush=True)
    time.sleep(300)
